In [1]:
import time
import json
import math
import os
import re

import numpy as np
import sentencepiece as spm
import torch
from transformers import AutoTokenizer

from custom_tokenizer import WhitespaceTokenizer
from train_lm import TrainConfig, build_model, evaluate

In [2]:
DATA_DIR = "data"
EVAL_BATCH_SIZE = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

# Eval documents are disjoint from the documents the tokenizers were trained on
with open("plwiki_eval.txt", encoding="utf-8") as f:
    eval_docs = [d for d in f.read().split("\n\n") if d.strip()]

val_texts = [line for doc in eval_docs for line in doc.split("\n") if line.strip()]
n_chars = sum(len(t) for t in val_texts)
n_words = sum(len(t.split()) for t in val_texts)

print(f"{len(val_texts)} lines, {n_words:,} words, {n_chars:,} characters")

21390 lines, 841,878 words, 6,193,473 characters


## Trained model loading

In [3]:
bpe = AutoTokenizer.from_pretrained("gpt2")
ws = WhitespaceTokenizer.load(os.path.join(DATA_DIR, "ws_tokenizer.json"))
sp = spm.SentencePieceProcessor(model_file=os.path.join(DATA_DIR, "sp.model"))

tokenizers = {
    "bpe": (bpe.encode, bpe.decode),
    "whitespace": (ws.encode, ws.decode),
    "sentencepiece": (sp.encode, sp.decode),
}


def load_model(name):
    run_dir = os.path.join("runs", name)
    with open(os.path.join(run_dir, "log.json")) as f:
        config = TrainConfig(**json.load(f)["config"])
    config.micro_batch_size = EVAL_BATCH_SIZE

    model = build_model(config).to(device)
    model.load_state_dict(
        torch.load(os.path.join(run_dir, "model.pt"), map_location=device)
    )
    return model.eval(), config


trained = [n for n in tokenizers if os.path.exists(os.path.join("runs", n, "model.pt"))]
print("trained models:", trained)

trained models: ['bpe', 'whitespace', 'sentencepiece']


In [4]:
# evaluate() returns nats per token over the evaluated blocks, so
# total loss = nats * n_tokens, and perplexity = exp(total loss / n_units)
results = {}
for name in trained:
    model, config = load_model(name)
    ids = np.load(os.path.join(DATA_DIR, f"ids_{name}_val.npy"))
    nats, _ = evaluate(model, ids, config, device, n_chars)
    results[name] = {
        "word_ppl": math.exp(nats * len(ids) / n_words),
        "char_ppl": math.exp(nats * len(ids) / n_chars),
    }
    del model
    torch.cuda.empty_cache()

print(f"{'tokenizer':<15}{'word ppl':>12}{'char ppl':>12}")
for name, r in results.items():
    print(f"{name:<15}{r['word_ppl']:>12.2f}{r['char_ppl']:>12.3f}")

tokenizer          word ppl    char ppl
bpe                 6672.11       3.310
whitespace           221.43       2.083
sentencepiece       1779.36       2.766


### The percentage of OOV words for whitespace tokenizer

In [5]:
# OOV words for the whitespace tokenizer: words only, punctuation tokens are not counted
words = [t for text in val_texts for t in ws.tokenize(text) if re.match(r"\w", t)]
oov = [t for t in words if t not in ws.token_to_id]
print(
    f"OOV word tokens: {len(oov):,} / {len(words):,} ({100 * len(oov) / len(words):.2f}%)"
)

types, oov_types = set(words), set(oov)
print(
    f"OOV word types: {len(oov_types):,} / {len(types):,} ({100 * len(oov_types) / len(types):.2f}%)"
)

OOV word tokens: 108,060 / 846,769 (12.76%)
OOV word types: 72,292 / 115,583 (62.55%)


In [6]:
# Efficiency: tokens per word / char, encoding speed (first ~1 MB of eval text), words kept as one token
sample, size = [], 0
for t in val_texts:
    if size >= 1_000_000:
        break
    sample.append(t)
    size += len(t)

eval_words = set(w for t in val_texts for w in re.findall(r"\w+", t))
space = {
    "bpe": " ",
    "whitespace": "",
    "sentencepiece": "",
}  # sentencepiece adds "▁" itself

print(
    f"{'tokenizer':<15}{'tok/word':>10}{'tok/char':>10}{'tok/s':>12}{'whole words':>14}"
)
for name, (encode, _) in tokenizers.items():
    n_tokens = len(np.load(os.path.join(DATA_DIR, f"ids_{name}_val.npy")))

    start = time.perf_counter()
    n_sample_tokens = sum(len(encode(t)) for t in sample)
    speed = n_sample_tokens / (time.perf_counter() - start)

    if name == "whitespace":  # encode() would map OOV words to a single <UNK> id
        whole = sum(w.lower() in ws.token_to_id for w in eval_words)
    else:
        whole = sum(len(encode(space[name] + w)) == 1 for w in eval_words)
    print(
        f"{name:<15}{n_tokens / n_words:>10.2f}{n_tokens / n_chars:>10.3f}"
        f"{speed:>12,.0f}{whole:>8,} ({100 * whole / len(eval_words):.1f}%)"
    )

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1254 > 1024). Running this sequence through the model will result in indexing errors


tokenizer        tok/word  tok/char       tok/s   whole words
bpe                  3.71     0.505     924,133   5,712 (4.4%)
whitespace           1.23     0.168   2,412,606  54,316 (42.2%)
sentencepiece        1.52     0.207   2,328,681  32,512 (25.3%)


In [7]:
@torch.no_grad()
def generate(name, prompt, n_new=40, temperature=0.8):
    encode, decode = tokenizers[name]
    model, config = load_model(name)
    ids = torch.tensor([encode(prompt)], device=device)
    for _ in range(n_new):
        logits = model(ids[:, -config.block_size :])[:, -1] / temperature
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())


torch.manual_seed(0)
for name in trained:
    print(name, "->", generate(name, "Warszawa jest"))

bpe -> Warszawa jest najpopularniejszym wybranym znem Bolesława Kłodzka.Podczas I wojny światowej w ok
whitespace -> warszawa jest siedzibą władz powiatowych. siedzibą wojewody jest kalisz. w katowicach mieści się również wojewódzki sąd okręgowy w warszawie w kielcach, składający się z 4 komisji publicznej rp. siedziba powiatu znajduje się na terenie powiatu. w latach
sentencepiece -> Warszawa jest siedzibą władz powiatu, które ma siedzibę w województwie białostockim. Siedzibą władz województwa jest powiat mielecki, który jest siedzibą powiatu wodnego z siedzibą w województwie tarnowskim. Siedzibą wojewody jest siedziba władz gromady Łyfiki


In [12]:
for name in trained:
    print(name, "->", generate(name, "Albert Einstein był"))

bpe -> Albert Einstein był najpowszechniejszym zainteresowaniem na: Guilty of Arts and Blues of Fame na świecie, Lake and Roll Hall of Fame
whitespace -> albert einstein był dwukrotnie żonaty. był drugim z jego <UNK> i jednym z jego kandydatury na rodzinę. dzieciństwo spędził w <UNK>, <UNK> się ze swoim synem. jego ojciec był starszym dzieckiem <UNK>. został pochowany na cmentarzu w château
sentencepiece -> Albert Einstein był jednym z organizatorów w Wielkiej Brytanii. W roku 1913 na krótko przed całej Polski pracował na Uniwersytecie Warszawskim, w którym Tracy otrzymał swój stopień. Zaczął przemówić służby do służby w armii wojskowej, na potrzeby drugiej


In [8]:
textA = "Czasem tylko w pogoni zaciekłe ogary, Wpadłszy niebacznie między bagna, mchy i jary, Wnętrznej ich okropności rażone widokiem, Uciekają skowycząc z obłąkanym wzrokiem; I długo potem ręką pana już głaskane, Drżą jeszcze u nóg jego strachem opętane. Te puszcze stołeczne, ludziom nieznane tajniki, W języku swoim strzelcy zowią: mateczniki."
textB = "Fotosynteza to proces, w którym rośliny, glony i niektóre bakterie przekształcają energię światła słonecznego w energię wiązań chemicznych. Z dwutlenku węgla i wody powstają wówczas cukry proste oraz tlen, który jest uwalniany do atmosfery. U roślin wyższych proces ten zachodzi w chloroplastach, zawierających zielony barwnik zwany chlorofilem."
textC = "Kraków jest jednym z najstarszych miast w Polsce i przez wiele wieków pełnił funkcję stolicy państwa. Jego historyczne centrum, z Rynkiem Głównym, Sukiennicami i zamkiem na Wawelu, zostało wpisane na listę światowego dziedzictwa UNESCO. Każdego roku miasto odwiedzają miliony turystów, a Uniwersytet Jagielloński, założony w 1364 roku, należy do najstarszych uczelni w Europie."

In [ ]:
token_strings = {
    "bpe": lambda t: bpe.convert_ids_to_tokens(bpe.encode(t)),
    "whitespace": lambda t: [ws.id_to_token[i] for i in ws.encode(t)],
    "sentencepiece": lambda t: sp.encode(t, out_type=str),
}

for label, text in {"A": textA, "B": textB, "C": textC}.items():
    words = re.findall(r"\w+", text)
    print(f"\n=== Text {label}: {len(text.split())} words ===")
    for name, (encode, _) in tokenizers.items():
        tokens = token_strings[name](text)
        if name == "whitespace":
            direct = sum(w.lower() in ws.token_to_id for w in words)
        else:
            direct = sum(len(encode(space[name] + w)) == 1 for w in words)
        print(
            f"\n{name}: {len(tokens) / len(text.split()):.2f} tokens/word, "
            f"{100 * direct / len(words):.0f}% words direct"
        )
        print(" | ".join(tokens))


=== Text A: 49 words ===

bpe: 3.94 tokens/word, 14% words direct
C | z | as | em | Ġty | l | ko | Ġw | Ġp | og | oni | Ġz | ac | ie | k | ÅĤ | e | Ġo | gary | , | ĠW | pad | ÅĤ | s | zy | Ġn | ie | b | ac | z | nie | Ġmi | Ä | Ļ | d | zy | Ġbag | na | , | Ġm | chy | Ġi | Ġj | ary | , | ĠW | n | Ä | Ļ | tr | z | ne | j | Ġ | ich | Ġok | rop | no | Å | Ľ | ci | Ġra | Å | ¼ | one | Ġwid | oki | em | , | ĠU | c | ie | k | aj | Ä | ħ | Ġsk | ow | y | cz | Ä | ħ | c | Ġz | Ġob | ÅĤ | Ä | ħ | k | any | m | Ġw | z | ro | ki | em | ; | ĠI | Ġd | ÅĤ | ug | o | Ġpot | em | Ġr | Ä | Ļ | k | Ä | ħ | Ġp | ana | Ġju | Å | ¼ | Ġg | ÅĤ | ask | ane | , | ĠDr | Å | ¼ | Ä | ħ | Ġj | es | z | c | ze | Ġu | Ġn | Ã³ | g | Ġj | eg | o | Ġstr | ac | hem | Ġop | Ä | Ļ | t | ane | . | ĠTe | Ġpus | z | c | ze | Ġsto | ÅĤ | ec | z | ne | , | Ġl | ud | zi | om | Ġn | ie | zn | ane | Ġt | aj | n | iki | , | ĠW | Ġj | Ä | Ļ | zy | ku | Ġsw | o | im | Ġstr | zel | cy | Ġz | ow | i | Ä | ħ | : | Ġmate | cz | n | iki 